In [40]:
############################################
# CELL 1 — IMPORTS
############################################

import os
import json
from datetime import datetime

import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer



############################################
# CELL 2 — SESSION + ROLE
############################################

session = sagemaker.Session()

role = sagemaker.get_execution_role()

print("Role:", role)



############################################
# CELL 3 — CREATE CODE FOLDER
############################################

os.makedirs("code", exist_ok=True)



############################################
# CELL 4 — CREATE TRAIN SCRIPT
############################################

training_script = """

import joblib
import os

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression


def main():

    iris = load_iris()

    X = iris.data
    y = iris.target

    model = LogisticRegression(max_iter=200)

    model.fit(X, y)

    model_dir = os.environ.get("SM_MODEL_DIR")

    joblib.dump(
        model,
        os.path.join(model_dir, "model.joblib")
    )


if __name__ == "__main__":

    main()

"""

with open("code/train.py", "w") as f:
    f.write(training_script)

print("train.py created")



############################################
# CELL 5 — CREATE INFERENCE SCRIPT
############################################

inference_script = """

import joblib
import json
import numpy as np
import os


def model_fn(model_dir):

    model = joblib.load(
        os.path.join(model_dir, "model.joblib")
    )

    return model



def input_fn(request_body, content_type):

    data = json.loads(request_body)

    return np.array([data])



def predict_fn(input_data, model):

    prediction = model.predict(input_data)

    return prediction



def output_fn(prediction, accept):

    return json.dumps(
        prediction.tolist()
    ), "application/json"

"""

with open("code/inference.py", "w") as f:
    f.write(inference_script)

print("inference.py created")



############################################
# CELL 6 — CONFIG
############################################

train_instance_type = "ml.m5.large"

inference_instance_type = "ml.m5.large"

endpoint_name = "iris-endpoint-" + datetime.utcnow().strftime("%H%M%S")

print("Endpoint:", endpoint_name)



############################################
# CELL 7 — TRAIN MODEL
############################################

estimator = SKLearn(

    entry_point="train.py",

    source_dir="code",

    framework_version="1.2-1",

    py_version="py3",

    role=role,

    instance_count=1,

    instance_type=train_instance_type

)

estimator.fit()



############################################
# CELL 8 — DEPLOY ENDPOINT
############################################

predictor = estimator.deploy(

    initial_instance_count=1,

    instance_type=inference_instance_type,

    endpoint_name=endpoint_name,

    entry_point="inference.py",

    source_dir="code"

)

predictor.serializer = JSONSerializer()

predictor.deserializer = JSONDeserializer()

print("Endpoint deployed")



############################################
# CELL 9 — TEST PREDICTION
############################################

result = predictor.predict([5.1, 3.5, 1.4, 0.2])

print("Prediction:", result)



############################################
# CELL 10 — DELETE ENDPOINT (SAVE MONEY)
############################################

# predictor.delete_endpoint()

# predictor.delete_model()

# print("Endpoint deleted")
print(region)

/tmp/ipykernel_7764/2209201792.py:142: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  endpoint_name = "iris-endpoint-" + datetime.utcnow().strftime("%H%M%S")
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Role: arn:aws:iam::480827623277:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
train.py created
inference.py created
Endpoint: iris-endpoint-103616


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-04-20-10-36-16-283


2026-04-20 10:36:17 Starting - Starting the training job...
2026-04-20 10:36:32 Starting - Preparing the instances for training...
2026-04-20 10:37:18 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-20 10:38:26,633 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-04-20 10:38:26,637 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-20 10:38:26,640 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-20 10:38:26,657 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-04-20 10:38:26,955 s

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-04-20-10-39-03-073
INFO:sagemaker:Creating endpoint-config with name iris-endpoint-103616
INFO:sagemaker:Creating endpoint with name iris-endpoint-103616


------!Endpoint deployed
Prediction: [0]
